# Initial clustering run

This notebook will contain the initial clustering run with one fixed aggregation level and window size,choosing of optimal k, saving the model.

In [18]:
# Main packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Clustering packages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Parallel processing packages
from joblib import Parallel, delayed

In [2]:
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [3]:
# Keep only human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

In [4]:
# Time conversions
SECONDS_IN_DAY = 60 * 60 * 24

In [ ]:
# FUNCTION - build features dataframe
def build_features(df,agg_hour_level):

    AGG_SECONDS = agg_hour_level * 60 * 60
    return (
        df.with_columns(
            bucket = pl.col('time') // AGG_SECONDS,
            theta = ((pl.col('time') % SECONDS_IN_DAY)/ SECONDS_IN_DAY) * 2 * np.pi,
            is_failure = (pl.col('outcome') == 'Fail').cast(pl.Int8),
        )
        .group_by(['src_user','bucket'])
        .agg(
            n_events = pl.len(),
            failure_ratio = pl.col('is_failure').mean(),
            n_distinct_dest = pl.col('dest_comp').n_unique(),
            n_distinct_src = pl.col('src_comp').n_unique(),
            c_bar = pl.col('theta').cos().mean(),
            s_bar = pl.col('theta').sin().mean(),
        )
        .with_columns(
             log_n_events=pl.col("n_events").log1p(),
             log_n_distinct_dest=pl.col("n_distinct_dest").log1p(),
             log_n_distinct_src=pl.col("n_distinct_src").log1p(),
        )
        .collect()
    )

In [ ]:
# Create features dataframe
agg_hour_level = 1
features_df = build_features(df,agg_hour_level)

In [ ]:
# Relevant feauture columns
feature_cols = [
    "log_n_events",
    "failure_ratio",
    "log_n_distinct_dest",
    "log_n_distinct_src",
    "c_bar",
    "s_bar",
]

In [ ]:
# FUNCTION - process features for clustering 
def cluster_preprocess(features_df,feature_cols,agg_hour_level,week):

    buckets_per_week = (7 * 24) // agg_hour_level
    lb = (week - 1) * buckets_per_week
    ub = lb + buckets_per_week - 1

    features_df = features_df.filter(pl.col('bucket').is_between(lb,ub))

    X = features_df.select(feature_cols).to_numpy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled

In [ ]:
# Process features
X_scaled = cluster_preprocess(features_df,feature_cols,agg_hour_level,week = 1)

In [19]:
# FUNCTION - kmeans 
def fit_kmeans(k, Y):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Y)
    score = silhouette_score(Y, labels, random_state=42)
    return k, score

In [ ]:
# Run kmeans in parallel
silhouette_scores = {}
results = Parallel(n_jobs=-1)(delayed(fit_kmeans)(k, X_scaled) for k in range(2, 9))
silhouette_scores = dict(results)

In [ ]:
# Silhoutte Plot 
plt.figure(figsize=(8, 4))
plt.plot(list(silhouette_scores.keys()), list(silhouette_scores.values()), marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("KMeans silhouette scores)
plt.xticks(list(range(2, 9)))
plt.tight_layout()
plt.show()

In [ ]:
# Find best k
best_k = max(silhouette_scores, key=silhouette_scores.get)
best_score = silhouette_scores[best_k]

# Refit on best k to get labels
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = km.fit_predict(X_scaled)

# Attach labels back to the dataframe
features_df = features_df.with_columns(
    pl.Series("cluster", labels)
)